In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "utils" / "paths.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook inside the project directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.paths import RAW, stage_path
from utils.runs import require_stage, stage_run
from loguru import logger
import csv
import random
from collections import Counter
from pathlib import Path

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.SimDivFilters.rdSimDivPickers import MaxMinPicker
from loguru import logger



In [ ]:
INPUT = require_stage("03_profiling") / "molecular_profiles.csv"



N_SELECT = 100
MAX_POOL_PER_GROUP = 5000
SEED = 42

COMMON_BRENK = {"Michael_acceptor_1", "beta-keto/anhydride"}
GROUPS = (
    "no_extra_no_pains",
    "pains_only",
    "extra_brenk_only",
    "both",
)

if not INPUT.is_file():
    raise FileNotFoundError(f"File does not exist: {INPUT.resolve()}")
logger.info("Source: {}", INPUT.resolve())

In [ ]:
def group_for(row):
    pains = str(row.get("PAINS", "")).strip().lower() in {"1", "true", "yes"}
    raw = row.get("Brenk_Details") or ""
    names = {part.strip() for part in raw.split(";") if part.strip()}
    has_extra = bool(names - COMMON_BRENK)

    if not pains and not has_extra:
        return "no_extra_no_pains"
    if pains and not has_extra:
        return "pains_only"
    if not pains and has_extra:
        return "extra_brenk_only"
    return "both"


def apportion(counts, size):
    total = sum(counts.values())
    if total == 0:
        return {g: 0 for g in GROUPS}

    raw = {g: size * counts[g] / total for g in GROUPS}
    quotas = {g: int(raw[g]) for g in GROUPS}
    missing = size - sum(quotas.values())
    order = sorted(
        GROUPS,
        key=lambda g: (raw[g] - quotas[g], counts[g]),
        reverse=True,
    )
    for g in order[:missing]:
        quotas[g] += 1
    return quotas

In [ ]:
with stage_run("04_selection", [INPUT], {"seed": SEED, "sample_size": N_SELECT, "pool_size": MAX_POOL_PER_GROUP}) as output_dir:
    OUTPUT_CSV = output_dir / "selected_ligands.csv"
    rng = random.Random(SEED)
    reservoirs = {g: [] for g in GROUPS}
    counts = Counter()
    seen_smiles = set()

    with INPUT.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        columns = list(reader.fieldnames or [])
        required = {"Name", "SMILES", "Status", "PAINS", "Brenk_Details"}
        missing = required - set(columns)
        if missing:
            raise ValueError(f"Missing CSV columns: {sorted(missing)}")

        for row in reader:
            if row.get("Status") != "OK":
                continue
            smi = row.get("CanonicalSMILES") or row.get("SMILES")
            if not smi or smi in seen_smiles:
                continue
            seen_smiles.add(smi)

            group = group_for(row)
            counts[group] += 1
            population = reservoirs[group]

            if len(population) < MAX_POOL_PER_GROUP:
                population.append(row)
            else:
                i = rng.randrange(counts[group])
                if i < MAX_POOL_PER_GROUP:
                    population[i] = row

    n = min(N_SELECT, sum(counts.values()))
    quotas = apportion(counts, n)

    logger.info(f"Unique structures with OK status: {sum(counts.values()):,}")
    logger.info(f"Target sample size: {n}")
    logger.info(f"{'Group':26} {'Library':>14} {'Pool':>8} {'Quota':>8}")
    for g in GROUPS:
        logger.info(f"{g:26} {counts[g]:14,} {len(reservoirs[g]):8,} {quotas[g]:8,}")
    fp_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    picker = MaxMinPicker()
    chosen_rows = []

    for idx, g in enumerate(GROUPS):
        population = reservoirs[g]
        if not population or quotas[g] == 0:
            continue

        usable, fingerprints = [], []
        for row in population:
            mol = Chem.MolFromSmiles(row.get("CanonicalSMILES") or row["SMILES"])
            if mol is not None:
                usable.append(row)
                fingerprints.append(fp_generator.GetFingerlogger.info(mol))

        quota = min(quotas[g], len(fingerprints))
        if quota == 0:
            continue

        chosen_indices = picker.LazyBitVectorPick(
            fingerprints, len(fingerprints), quota, seed=SEED + idx
        )
        for j in chosen_indices:
            row = dict(usable[j])
            row["SelectionGroup"] = g
            row["SelectionMethod"] = "stratified_reservoir_Morgan2_MaxMin"
            row["SelectionSeed"] = SEED
            chosen_rows.append(row)

    if len(chosen_rows) != n:
        raise RuntimeError(f"Expected {n} molecules, received {len(chosen_rows)}")

    assert len({r.get("CanonicalSMILES") or r["SMILES"] for r in chosen_rows}) == len(chosen_rows)
    logger.info(f"Selected unique molecules: {len(chosen_rows)}")
    for g in GROUPS:
        logger.info(f"  {g}: {sum(r['SelectionGroup'] == g for r in chosen_rows)}")
    fields = columns + ["SelectionGroup", "SelectionMethod", "SelectionSeed"]
    with OUTPUT_CSV.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(chosen_rows)

    logger.success("Selection saved")
    logger.info("CSV: {}", OUTPUT_CSV.resolve())


In [ ]:
from pprint import pprint

for row in chosen_rows[:5]:
    pprint({
        "Name": row["Name"],
        "SelectionGroup": row["SelectionGroup"],
        "MW": row.get("MW"),
        "TPSA": row.get("TPSA"),
        "cLogP": row.get("cLogP"),
        "QED": row.get("QED"),
        "SMILES": row.get("CanonicalSMILES") or row["SMILES"],
    })